In [5]:
import requests
import csv
import pandas as pd
import math
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import time
import random
from datetime import datetime, date
import statistics
import requests
from scipy.stats import skew, kurtosis
from scipy import stats
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from scipy import stats
from scipy.stats import johnsonsu
from scipy.optimize import minimize
from scipy.stats import norm

In [6]:
def get_data(ticker, size='full', date_filter=None):
    """ Retrieve time series data for the price of a security in data frame format
        Sorts so that first date is first index, filters on date if given, renames columns and converts datatypes

    Args:
        ticker (string): The ticker for the security to pull data from
        size (string): 'full' since 1999, 'compact' is last 100 trading days
        date_filter (date string): Filters  data to  keep data only the filter date onward

    Returns:
        df: A dataframe with open, close, high, low, adj close price, volume, split coef, date
    """
    key='&apikey=ZKMMTO1ATDBLXH2K' # API Key
    api_ticker=f'&symbol={ticker}' # Ticker
    endpoint='function=TIME_SERIES_DAILY_ADJUSTED' # Called 'function', the dataset we want
    size=f'&outputsize={size}'
    web='https://www.alphavantage.co/query?'
    url =web+endpoint+api_ticker+size+key

    r = requests.get(url)
    # print(r.status_code) # 200 good, 400 bad
    data = r.json()

    # print(data.keys()) #printing the keys
    meta = data['Meta Data']
    time_series_data = data['Time Series (Daily)']

    ts_df = pd.DataFrame.from_dict(time_series_data, orient='index').reset_index().rename(columns={'index': 'Date'})
    clean_cols_dict = {'1. open': 'Open', '2. high': 'High', '3. low': 'Low', '4. close': 'Close', # Dictionary to convert the names of the columns
                '5. adjusted close': 'Adj Close', '6. volume': 'Volume', '7. dividend amount': 'Dividend', '8. split coefficient': 'Split Coef'}
    ts_df = ts_df.rename(columns=clean_cols_dict)

    ts_df['Date'] = pd.to_datetime(ts_df['Date'])
    if date_filter is not None:
        date_filter = pd.to_datetime(date_filter)
        ts_df = ts_df[ts_df['Date'] >= date_filter]


    ts_df = ts_df.sort_values(by='Date', ascending=True).reset_index().drop(columns='index')
    ts_df['Adj Close'] = ts_df['Adj Close'].astype('float').round(4)
    ts_df['Ticker'] = ticker
    
    return ts_df

In [7]:
def calculate_returns(df, frequency=1):
    """ Calculate the log returns of a security given a df with its prices over a time period

    Args:
        df (dataframe): A dataframe with columns Date, Ticker, Volume, and Adj Close
        frequency (int, optional): How often you want to calculate a return Defaults to 1.

    Returns:
        df: The dataframe from the start with an additional 'Log Return' column which is the log return over the interval for each row
    """
    returns_df = df[['Date', 'Ticker', 'Volume', 'Adj Close']].copy()
    returns_df['Adj Close'] = returns_df['Adj Close'].astype('float')
    returns_df['Log Return'] = np.log(
        returns_df['Adj Close'] / returns_df['Adj Close'].shift(frequency)
    )
    returns_df = returns_df.dropna()

    # If frequency > 1, keep only every `frequency`-th row
    if frequency > 1:
        returns_df = returns_df.iloc[frequency-1::frequency].reset_index(drop=True)

    return returns_df



In [10]:
qqq_df = get_data('QQQ', size='full', date_filter='2021-01-01')
qqq_df = qqq_df[qqq_df['Date'] <= pd.to_datetime('2026-01-01')]
returns_df = calculate_returns(qqq_df, frequency=1).rename(columns={'Adj Close': 'Price'})
returns_df

,Date,Ticker,Volume,Price,Log Return
1,2021-01-05,QQQ,29323409,302.2911,0.008211
2,2021-01-06,QQQ,52809622,298.1036,-0.013949
3,2021-01-07,QQQ,30394826,305.3154,0.023904
4,2021-01-08,QQQ,33955847,309.2411,0.012776
5,2021-01-11,QQQ,32869095,304.7725,-0.014556
...,...,...,...,...,...
1250,2025-12-24,QQQ,18468707,623.1534,0.002921
1251,2025-12-26,QQQ,28959778,623.1134,-0.000064
1252,2025-12-29,QQQ,32458308,620.0972,-0.004852
1253,2025-12-30,QQQ,31226754,618.6590,-0.002322


In [11]:
# MLE for Log Return distribution: Normal vs Student-t
r = returns_df['Log Return'].dropna().values
n = len(r)


def neg_log_likelihood(params, data, dist):
    """Negative log-likelihood (what we minimize = maximize likelihood)."""
    if dist == 'normal':
        mu, sigma = params
        if sigma <= 0:
            return np.inf
        return -np.sum(stats.norm.logpdf(data, loc=mu, scale=sigma))

    if dist == 't':
        df, mu, sigma = params
        if df <= 2 or sigma <= 0:  # df>2 keeps variance finite; sigma must be positive
            return np.inf
        return -np.sum(stats.t.logpdf(data, df=df, loc=mu, scale=sigma))


# --- Model 1: Normal ---
mu0, sigma0 = r.mean(), r.std(ddof=0)  # ddof=0 matches MLE (divide by n, not n-1)
normal_start = [mu0, sigma0]
normal_fit = minimize(
    neg_log_likelihood, normal_start, args=(r, 'normal'),
    method='L-BFGS-B', bounds=[(None, None), (1e-8, None)]
)
mu_hat, sigma_hat = normal_fit.x
normal_ll = -neg_log_likelihood(normal_fit.x, r, 'normal')

# --- Model 2: Student-t (fatter tails than normal) ---
t_start = [5.0, mu0, sigma0]  # df=5 is a reasonable starting guess for equity returns
t_fit = minimize(
    neg_log_likelihood, t_start, args=(r, 't'),
    method='L-BFGS-B', bounds=[(2.01, None), (None, None), (1e-8, None)]
)
df_hat, mu_t, sigma_t = t_fit.x
t_ll = -neg_log_likelihood(t_fit.x, r, 't')


def info_criteria(log_likelihood, n_params, n_obs):
    """AIC/BIC: lower = better fit penalizing extra parameters."""
    aic = 2 * n_params - 2 * log_likelihood
    bic = n_params * np.log(n_obs) - 2 * log_likelihood
    return aic, bic


normal_aic, normal_bic = info_criteria(normal_ll, 2, n)
t_aic, t_bic = info_criteria(t_ll, 3, n)

# KS test: how close empirical data is to each fitted CDF (higher p = closer match)
ks_normal = stats.kstest(r, 'norm', args=(mu_hat, sigma_hat))
ks_t = stats.kstest(r, 't', args=(df_hat, mu_t, sigma_t))

print("=" * 55)
print("MODEL 1: NORMAL")
print(f"  mu    = {mu_hat:.6f}")
print(f"  sigma = {sigma_hat:.6f}")
print(f"  daily vol (%)     = {sigma_hat * 100:.4f}")
print(f"  annualized vol (%)= {sigma_hat * np.sqrt(252) * 100:.2f}")
print(f"  log-likelihood = {normal_ll:.2f}")
print(f"  AIC = {normal_aic:.2f}   BIC = {normal_bic:.2f}")
print(f"  KS p-value = {ks_normal.pvalue:.4f}")

print("\n" + "=" * 55)
print("MODEL 2: STUDENT-t")
print(f"  df (tail heaviness, lower = fatter tails) = {df_hat:.2f}")
print(f"  mu    = {mu_t:.6f}")
print(f"  sigma = {sigma_t:.6f}")
print(f"  log-likelihood = {t_ll:.2f}")
print(f"  AIC = {t_aic:.2f}   BIC = {t_bic:.2f}")
print(f"  KS p-value = {ks_t.pvalue:.4f}")

print("\n" + "=" * 55)
print("COMPARISON (lower AIC/BIC is better)")
winner_aic = "Student-t" if t_aic < normal_aic else "Normal"
winner_bic = "Student-t" if t_bic < normal_bic else "Normal"
print(f"  AIC winner: {winner_aic}")
print(f"  BIC winner: {winner_bic}")
print(f"  Sample skew = {stats.skew(r):.3f}, excess kurtosis = {stats.kurtosis(r):.3f}")
print("  (Positive excess kurtosis -> fat tails; t often beats normal)")

MODEL 1: NORMAL
  mu    = 0.000571
  sigma = 0.014261
  daily vol (%)     = 1.4261
  annualized vol (%)= 22.64
  log-likelihood = 3550.41
  AIC = -7096.83   BIC = -7086.56
  KS p-value = 0.0000

MODEL 2: STUDENT-t
  df (tail heaviness, lower = fatter tails) = 5.00
  mu    = 0.001033
  sigma = 0.010958
  log-likelihood = 3611.95
  AIC = -7217.90   BIC = -7202.50
  KS p-value = 0.3824

COMPARISON (lower AIC/BIC is better)
  AIC winner: Student-t
  BIC winner: Student-t
  Sample skew = 0.030, excess kurtosis = 4.462
  (Positive excess kurtosis -> fat tails; t often beats normal)
